### Imports 

In [ ]:
from pathlib import Path
import geopandas as gpd
import numpy as np
from shapely import make_valid
import pandas as pd

### Base and output paths

In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis")

out_path = base_path / "major_basins_plus_coastal_unionize_small.gpkg"


## Old catchments with small ones included

In [ ]:
major_basins_plus_coastal = base_path / "major_basins_plus_coastal.gpkg"
major_basins_plus_coastal = gpd.read_file(major_basins_plus_coastal)

major_basins_plus_coastal.head()

In [ ]:
catchments_gdf = major_basins_plus_coastal.copy()

# 0) One catchment per polygon, numeric ID 1..N
catchments_gdf = (
    major_basins_plus_coastal
      .loc[major_basins_plus_coastal.geometry.notna(), ["geometry"]]
      .reset_index(drop=True)
      .copy()
)
catchments_gdf["catchment_uid"] = catchments_gdf.index + 1


id_field = "catchment_uid"

catchments_gdf
print("Input feature count:", len(catchments_gdf))  # should be your original count (you saw 94)

# Specify number of pixels to set as minimum threshold

In [ ]:
# --- Define what "tiny" means (based on DEM resolution × pixel count) -------
dem_pixel_size_m = 30.0     # e.g., 10.0 or 30.0
tiny_pixel_count = 95        # merge polygons up to 3 pixels in area
tiny_area_threshold_m2 = dem_pixel_size_m**2 * tiny_pixel_count



In [ ]:
# --- Outputs ----------------------------------------------------------------
output_gpkg_path = base_path / "major_basins_plus_coastal_unionize_small.gpkg"
output_layer_name = "catchments_small_merged"
merge_mapping_csv_path = base_path / "major_basins_plus_coastal_merge_small_catchments_map.csv"



In [ ]:
# --- Helpers ----------------------------------------------------------------
def choose_best_neighbor_index(small_geom, neighbor_gdf):
    """
    Return index of the neighbor sharing the longest boundary with small_geom.
    If all shared-border lengths are zero (only corner touches), fall back to nearest centroid.
    """
    small_boundary = small_geom.boundary
    shared_lengths = []
    for j, g2 in zip(neighbor_gdf.index, neighbor_gdf.geometry):
        L = float(small_boundary.intersection(g2.boundary).length)
        shared_lengths.append((j, L))

    if shared_lengths:
        j_best, L_best = max(shared_lengths, key=lambda t: t[1])
        if L_best > 0:
            return j_best

    # Fallback: nearest centroid among the candidate neighbors
    c0 = small_geom.centroid
    dists = [(j, float(c0.distance(g2.centroid))) for j, g2 in zip(neighbor_gdf.index, neighbor_gdf.geometry)]
    j_best, _ = min(dists, key=lambda t: t[1])
    return j_best

def collapse_merge_chains(idx_map):
    """
    Resolve chains like A->B, B->C into A->C so each tiny feature ultimately
    points to a stable 'root' neighbor index.
    """
    def root(i):
        seen = set()
        cur = i
        while cur in idx_map and cur not in seen:
            seen.add(cur)
            cur = idx_map[cur]
        return cur
    return {k: root(k) for k in idx_map.keys()}


In [ ]:
# --- Prep: compute area; no geometry fixing, no explode ---------------------
# Assumes your CRS units are meters (you said CRS is fine)
catchments_gdf["area_m2"] = catchments_gdf.geometry.area

# --- Identify tiny catchments ------------------------------------------------
tiny_mask = catchments_gdf["area_m2"] <= tiny_area_threshold_m2
tiny_catchments = catchments_gdf.loc[tiny_mask]
print(f"Total polygons: {len(catchments_gdf):,}  |  Tiny (≤ {tiny_area_threshold_m2:,.0f} m²): {len(tiny_catchments):,}")

if tiny_catchments.empty:
    print("No tiny polygons found. Writing a copy and exiting.")
    catchments_gdf.to_file(output_gpkg_path, driver="GPKG", layer=output_layer_name)
else:
    # Build spatial index once
    catchments_sindex = catchments_gdf.sindex

    # Map: tiny feature .index -> chosen neighbor .index
    tiny_to_neighbor_index_map = {}

    # Process smallest first for stability
    for i in tiny_catchments.sort_values("area_m2").index:
        geom_i = catchments_gdf.at[i, "geometry"]

        # Find bbox candidates, then require touch/intersect to be valid neighbors
        candidate_idxs = list(catchments_sindex.query(geom_i.buffer(0.01)))  # tiny buffer for robustness
        candidate_idxs = [j for j in candidate_idxs if j != i]
        if not candidate_idxs:
            # no neighbors found; leave as-is
            continue

        neighbor_candidates = catchments_gdf.loc[candidate_idxs]
        touching_or_intersecting = neighbor_candidates[
            neighbor_candidates.geometry.touches(geom_i) | neighbor_candidates.geometry.intersects(geom_i)
        ]
        if touching_or_intersecting.empty:
            touching_or_intersecting = neighbor_candidates  # fallback to bbox candidates

        best_neighbor_idx = choose_best_neighbor_index(geom_i, touching_or_intersecting)
        tiny_to_neighbor_index_map[i] = best_neighbor_idx

    # Collapse chains and determine each feature's final target index
    chain_roots = collapse_merge_chains(tiny_to_neighbor_index_map)
    target_index_for_row = {i: (chain_roots[i] if i in chain_roots else i) for i in catchments_gdf.index}

    # Stable group key = target row's original catchment_uid
    catchments_gdf["_merge_group_id"] = [catchments_gdf.at[target_index_for_row[i], id_field]
                                         for i in catchments_gdf.index]

    # Save old -> new ID mapping (one row per original feature)
    merge_mapping = (
        catchments_gdf[[id_field, "_merge_group_id"]]
        .drop_duplicates()
        .rename(columns={id_field: "old_id", "_merge_group_id": "new_id"})
        .sort_values(["old_id", "new_id"])
    )
    merge_mapping.to_csv(merge_mapping_csv_path, index=False)
    print(f"Saved merge mapping: {merge_mapping_csv_path}")

    # --- Build aggregation spec WITHOUT summing or carrying the ID/group columns
    non_geom_cols = [c for c in catchments_gdf.columns if c != "geometry"]
    
    # numeric columns (exclude ID/group/area)
    numeric_cols = catchments_gdf[non_geom_cols].select_dtypes(include="number").columns.tolist()
    exclude_from_sum = {id_field, "_merge_group_id", "area_m2"}
    numeric_cols = [c for c in numeric_cols if c not in exclude_from_sum]
    
    # other (non-numeric) columns, but exclude the ID and the grouping key
    other_cols = [c for c in non_geom_cols if c not in numeric_cols and c not in {id_field, "_merge_group_id"}]
    
    agg_spec = {c: "sum" for c in numeric_cols}
    for c in other_cols:
        agg_spec[c] = "first"
    
    # Dissolve and save
    merged_catchments = catchments_gdf.dissolve(by="_merge_group_id", aggfunc=agg_spec, as_index=False)
    
    # After dissolve, the only ID should be the group key; rename it to your id_field
    merged_catchments = merged_catchments.rename(columns={"_merge_group_id": id_field})
    
    # Recompute area on merged geometries
    merged_catchments["area_m2"] = merged_catchments.geometry.area
    
    # (Safety) ensure no duplicate column names before writing
    if merged_catchments.columns.duplicated().any():
        dupes = merged_catchments.columns[merged_catchments.columns.duplicated()].tolist()
        raise ValueError(f"Duplicate columns remain: {dupes}")
    
    merged_catchments.to_file(output_gpkg_path, driver="GPKG", layer=output_layer_name)
    print(f"Saved merged layer: {output_gpkg_path} (layer={output_layer_name}, {len(merged_catchments):,} polygons)")

In [ ]:
# Check min/max areas after merge
print("Min area (m²):", float(merged_catchments["area_m2"].min()))
print("Max area (m²):", float(merged_catchments["area_m2"].max()))
print("Any < threshold?",
      bool((merged_catchments["area_m2"] < tiny_area_threshold_m2).any()))

In [ ]:
# --- Report areas in km² ---
area_km2 = merged_catchments["area_m2"] / 1_000_000
threshold_km2 = tiny_area_threshold_m2 / 1_000_000

print(f"Min area: {area_km2.min():,.6f} km²")
print(f"Max area: {area_km2.max():,.6f} km²")
print(f"Any < threshold ({threshold_km2:,.6f} km²)? {bool((area_km2 < threshold_km2).any())}")
print(f"Count < threshold: {(area_km2 < threshold_km2).sum()} / {len(area_km2)}")

# (optional) keep the km² in the GeoDataFrame
merged_catchments["area_km2"] = area_km2

In [ ]:
# --- Summary stats in km² (and pixel equivalents) ---------------------------
# assumes you already defined: area_km2, tiny_area_threshold_m2, dem_pixel_size_m

cell_area_m2 = dem_pixel_size_m ** 2     # e.g., 30 m → 900 m² per pixel
threshold_km2 = tiny_area_threshold_m2 / 1_000_000

s = area_km2  # shorthand

stats = {
    "count": len(s),
    "min_km2": s.min(),
    "p05_km2": s.quantile(0.05),
    "q1_km2":  s.quantile(0.25),
    "median_km2": s.median(),
    "mean_km2": s.mean(),
    "q3_km2":  s.quantile(0.75),
    "p95_km2": s.quantile(0.95),
    "max_km2": s.max(),
    "std_km2": s.std(),
    "total_km2": s.sum(),
}

# Pixel equivalents (approx.)
to_px = lambda km2: (km2 * 1_000_000) / cell_area_m2
min_px     = to_px(stats["min_km2"])
median_px  = to_px(stats["median_km2"])
mean_px    = to_px(stats["mean_km2"])

print(f"Features: {stats['count']:,}")
print(f"Min:      {stats['min_km2']:,.6f} km²  (~{min_px:.2f} px)")
print(f"P05:      {stats['p05_km2']:,.6f} km²")
print(f"Q1:       {stats['q1_km2']:,.6f} km²")
print(f"Median:   {stats['median_km2']:,.6f} km²  (~{median_px:.2f} px)")
print(f"Mean:     {stats['mean_km2']:,.6f} km²  (~{mean_px:.2f} px)")
print(f"Q3:       {stats['q3_km2']:,.6f} km²")
print(f"P95:      {stats['p95_km2']:,.6f} km²")
print(f"Max:      {stats['max_km2']:,.6f} km²")
print(f"Std.dev:  {stats['std_km2']:,.6f} km²")
print(f"Total:    {stats['total_km2']:,.3f} km²")
print(f"Threshold: {threshold_km2:,.6f} km²  | Any < thr? {bool((s < threshold_km2).any())} "
      f"| Count < thr: {(s < threshold_km2).sum()} / {len(s)}")

# (optional) add hectares & pixels as columns for later use
merged_catchments["area_km2"] = s
merged_catchments["area_ha"] = merged_catchments["area_m2"] / 10_000
merged_catchments["area_pixels_est"] = ((merged_catchments["area_m2"]) / cell_area_m2).round(2)

In [ ]:
# Show top 5 smallest by area (km²) with more precision
print("\nTop 30 smallest catchments (km², 8 d.p.):")
print(
    merged_catchments[["catchment_uid","area_km2"]]
      .sort_values("area_km2")
      .head(30)
      .assign(area_km2=lambda d: d["area_km2"].map(lambda x: f"{x:.8f}"))
      .to_string(index=False)
)

In [ ]:
# --- Renumber: keep old as `catchment_uid_old`, set new `catchment_uid` = 1..N ---

merged_catchments = merged_catchments.copy()

# 1) Preserve old/root IDs from the dissolve
if "catchment_uid_old" not in merged_catchments.columns:
    merged_catchments["catchment_uid_old"] = merged_catchments["catchment_uid"]

# 2) Deterministic ordering (south→north, then west→east) for stable numbering
cent = merged_catchments.geometry.centroid
merged_catchments = (
    merged_catchments.assign(cx=cent.x, cy=cent.y)
    .sort_values(["cy", "cx"])
    .drop(columns=["cx", "cy"])
    .reset_index(drop=True)
)

# 3) New sequential IDs starting at 1
merged_catchments["catchment_uid"] = np.arange(1, len(merged_catchments) + 1, dtype=int)
print("New ID range:", merged_catchments["catchment_uid"].min(), "→", merged_catchments["catchment_uid"].max())

# 4) Compose mapping: original -> new sequential
#    (you already saved original -> root as `merge_mapping_csv_path`)
old_to_root = pd.read_csv(merge_mapping_csv_path).rename(columns={"new_id": "root_id"})  # cols: old_id, root_id
root_to_seq = merged_catchments[["catchment_uid_old", "catchment_uid"]].rename(
    columns={"catchment_uid_old": "root_id", "catchment_uid": "new_id"}
)
old_to_seq = old_to_root.merge(root_to_seq, on="root_id", how="left")[["old_id", "new_id"]]

# 5) Save final GPKG (single layer "catchments") and crosswalk CSV
catchments_unionized_final = base_path / "major_basins_plus_coastal_unionized_final.gpkg"
crosswalk_final_csv        = base_path / "major_basins_plus_coastal_id_crosswalk_final.csv"

merged_catchments.to_file(catchments_unionized_final, driver="GPKG", layer="catchments")
old_to_seq.to_csv(crosswalk_final_csv, index=False)

print(f"Wrote GPKG: {catchments_unionized_final} (layer='catchments', features={len(merged_catchments):,})")
print(f"Wrote crosswalk: {crosswalk_final_csv}")


In [ ]:
merged_catchments